In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import requests
import re
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")


: 

In [ ]:
print("📊 Loading data...")
df = pd.read_csv("../data/nycparking2025.csv")
print(f"✅ Loaded {len(df):,} rows and {len(df.columns)} columns")

print("\n📋 Column names:")
for i, col in enumerate(df.columns):
    print(f"   {i+1}. {col}")

print("\n👀 First 5 rows preview:")
df.head()

In [ ]:
# Make column names easier to work with
df.columns = [col.replace(' ', '_').lower() for col in df.columns]

# Create a clean copy
df_clean = df.copy()

# Convert date column
df_clean['issue_date'] = pd.to_datetime(df_clean['issue_date'], errors='coerce')

# Clean text columns (uppercase and strip whitespace)
text_columns = ['plate_id', 'registration_state', 'vehicle_color', 'vehicle_make', 
                'vehicle_body_type', 'violation_description', 'street_name']

for col in text_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.upper().str.strip()
        df_clean[col] = df_clean[col].replace(['NAN', 'NONE', ''], pd.NA)

# Remove rows without a summons number (primary key)
df_clean = df_clean.dropna(subset=['summons_number'])

print(f"✅ Cleaned data: {len(df_clean):,} rows")
print(f"📅 Date range: {df_clean['issue_date'].min()} to {df_clean['issue_date'].max()}")

In [ ]:
 #Load your extracted fines CSV
df_fines_raw = pd.read_csv("../data/fines_extracted_fixed.csv")

print("=" * 60)
print("INSPECTING EXTRACTED FINES DATA")
print("=" * 60)

print(f"\n📊 Shape: {df_fines_raw.shape}")
print(f"\n📋 Columns: {df_fines_raw.columns.tolist()}")
print(f"\n👀 First 10 rows:")
print(df_fines_raw.head(10))
print(f"\n🔍 Data types:")
print(df_fines_raw.dtypes)
print(f"\n📊 Missing values:")
print(df_fines_raw.isnull().sum())
print(df_fines_raw.head(20))

In [ ]:
print("=" * 60)
print("CLEANING COMPLETE FINES DATA")
print("=" * 60)

# Load the complete fines CSV
# Note: Your column name has a trailing space 'Violation '
df_fines = pd.read_csv("../data/fines_extracted_fixed.csv")

# Fix column names (remove trailing spaces)
df_fines.columns = df_fines.columns.str.strip()

print(f"✅ Cleaned column names: {df_fines.columns.tolist()}")
print(f"📊 Total rows in fines file: {len(df_fines):,}")

# Clean the Fine Amount column (remove $ and convert to number)
df_fines['fine_amount_clean'] = df_fines['Fine Amount'].astype(str).str.replace('$', '').str.strip()
df_fines['fine_amount_clean'] = pd.to_numeric(df_fines['fine_amount_clean'], errors='coerce')

# Clean violation code column
df_fines['violation_code'] = df_fines['Violation'].astype(str).str.strip()

# Remove any rows with missing fine amounts
df_fines = df_fines.dropna(subset=['fine_amount_clean'])

print(f"\n✅ After cleaning: {len(df_fines):,} violation codes with fines")
print(f"\n📊 Fine amount statistics:")
print(df_fines['fine_amount_clean'].describe())
print(f"\n📊 Fine amount distribution:")
print(df_fines['fine_amount_clean'].value_counts().sort_index().head(20))

In [ ]:
print("=" * 60)
print("CHECKING PARKING DATA VIOLATION CODES")
print("=" * 60)

# How many unique violation codes in your parking data
parking_codes = df_clean['violation_code'].astype(str).str.strip().nunique()
print(f"📊 Unique violation codes in parking data: {parking_codes:,}")

# Show sample of parking violation codes
print(f"\n📋 Sample of violation codes in parking data:")
print(df_clean['violation_code'].astype(str).str.strip().value_counts().head(20))

In [ ]:
print("🔍 SCANNING FOR COLUMNS WITH REAL DATA")
print("=" * 60)

# List of all columns in your dataset
all_columns = ['summons_number', 'plate_id', 'registration_state', 'plate_type', 'issue_date',
               'violation_code', 'vehicle_body_type', 'vehicle_make', 'issuing_agency',
               'street_code1', 'street_code2', 'street_code3', 'vehicle_expiration_date',
               'violation_location', 'violation_precinct', 'issuer_precinct', 'issuer_code',
               'issuer_command', 'issuer_squad', 'violation_time', 'time_first_observed',
               'violation_county', 'violation_in_front_of_or_opposite', 'house_number',
               'street_name', 'intersecting_street', 'date_first_observed', 'law_section',
               'sub_division', 'violation_legal_code', 'days_parking_in_effect',
               'from_hours_in_effect', 'to_hours_in_effect', 'vehicle_color',
               'unregistered_vehicle?', 'vehicle_year', 'meter_number', 'feet_from_curb',
               'violation_post_code', 'violation_description', 'no_standing_or_stopping_violation',
               'hydrant_violation', 'double_parking_violation']

useful_columns = []

print("\n📊 COLUMNS WITH MEANINGFUL DATA:")
print("-" * 40)

for col in all_columns:
    if col in df_clean.columns:
        non_null_count = df_clean[col].notna().sum()
        non_null_pct = (non_null_count / len(df_clean)) * 100
        unique_count = df_clean[col].nunique()
        
        # Determine if column has useful data (more than 1% non-null)
        if non_null_pct > 1:
            useful_columns.append(col)
            print(f"\n✅ {col}")
            print(f"   Non-null: {non_null_count:,} rows ({non_null_pct:.1f}%)")
            print(f"   Unique values: {unique_count:,}")
            
            # Show sample values for context
            sample_values = df_clean[col].dropna().head(3).tolist()
            print(f"   Sample: {sample_values}")
        else:
            print(f"\n❌ {col} - Too few values ({non_null_pct:.1f}% non-null)")

print("\n" + "=" * 60)
print(f"📈 SUMMARY: {len(useful_columns)} columns have meaningful data out of {len(all_columns)}")
print("=" * 60)

In [ ]:
# ============================================
# CLEAN INVALID FUTURE DATES
# Run this BEFORE any time-based analysis
# ============================================

print("📅 Cleaning invalid future dates...")

# Store original row count
original_count = len(df_clean)

# Ensure issue_date is datetime type
df_clean['issue_date'] = pd.to_datetime(df_clean['issue_date'], errors='coerce')

# Get today's date
today = pd.Timestamp.now().normalize()

# Remove future dates (anything after today)
df_clean = df_clean[df_clean['issue_date'] <= today]

# Also remove dates that are unreasonably old (before 2000)
# Parking violation data in NYC is reliable from 2000 onward
df_clean = df_clean[df_clean['issue_date'] >= '2000-01-01']

# Remove rows where date conversion failed (NaT)
df_clean = df_clean.dropna(subset=['issue_date'])

# Count removed rows
removed_count = original_count - len(df_clean)
removed_percent = (removed_count / original_count) * 100

print(f"✅ Removed {removed_count:,} rows with invalid dates ({removed_percent:.1f}% of data)")
print(f"✅ Remaining rows: {len(df_clean):,}")
print(f"📅 New date range: {df_clean['issue_date'].min().date()} to {df_clean['issue_date'].max().date()}")

# Double-check no future dates remain
future_check = df_clean[df_clean['issue_date'] > today]
if len(future_check) == 0:
    print("✅ Verification: No future dates remain in dataset")
else:
    print(f"⚠️ Warning: {len(future_check)} future dates still present")

In [ ]:
print("👮 Issuing Agency Breakdown:")

if 'issuing_agency' in df_clean.columns:
    agency_counts = df_clean['issuing_agency'].value_counts().head(10)
    
    plt.figure(figsize=(10, 6))
    agency_counts.plot(kind='bar', color='darkgreen', edgecolor='black')
    plt.xlabel('Agency', fontsize=12)
    plt.ylabel('Number of Tickets', fontsize=12)
    plt.title('Top 10 Issuing Agencies for Parking Violations', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print(agency_counts)

In [ ]:
print("🚓 Top 10 Precincts for Parking Violations:")

top_precincts = df_clean['violation_precinct'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_precincts.plot(kind='bar', color='coral', edgecolor='black')
plt.xlabel('Precinct Number', fontsize=12)
plt.ylabel('Number of Tickets', fontsize=12)
plt.title('Top 10 NYPD Precincts for Parking Violations', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(top_precincts)

In [ ]:
print("🔍 Checking for violation flag columns:")

# Look for columns that might indicate specific violation types
flag_columns = ['hydrant_violation', 'double_parking_violation', 'no_standing_or_stopping_violation']

for col in flag_columns:
    if col in df_clean.columns:
        values = df_clean[col].value_counts()
        non_null_count = df_clean[col].notna().sum()
        print(f"\n📋 {col}:")
        print(f"   Non-null values: {non_null_count:,} rows")
        if non_null_count > 0:
            print(f"   Value counts: {values.to_dict()}")
        else:
            print(f"   ℹ️ Column exists but all values are null")
    else:
        print(f"\n📋 {col}: Not found in dataset")

In [ ]:
print("📍 Top Streets for Parking Violations:")

if 'street_name' in df_clean.columns:
    top_streets = df_clean['street_name'].value_counts().head(10)
    
    plt.figure(figsize=(12, 6))
    top_streets.plot(kind='barh', color='purple', edgecolor='black')
    plt.xlabel('Number of Tickets', fontsize=12)
    plt.ylabel('Street Name', fontsize=12)
    plt.title('Top 10 Streets for Parking Violations', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(top_streets)

In [ ]:
print("🔍 Missing Values Analysis:")
missing_counts = df_clean.isnull().sum()
missing_percent = (missing_counts / len(df_clean)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percent (%)': missing_percent
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print(missing_df.head(15))
else:
    print("✅ No missing values found!")

print(f"\n📊 Data Types:")
print(df_clean.dtypes.value_counts())

In [ ]:
print("🚨 Top 10 Parking Violations:")

top_violations = df_clean.groupby(['violation_code', 'violation_description']).size().reset_index(name='ticket_count')
top_violations = top_violations.sort_values('ticket_count', ascending=False).head(10)
top_violations

In [ ]:
plt.figure(figsize=(12, 6))
bars = plt.barh(range(len(top_violations)), top_violations['ticket_count'])
plt.yticks(range(len(top_violations)), top_violations['violation_description'].str[:50])
plt.xlabel('Number of Tickets', fontsize=12)
plt.title('Top 10 Parking Violations in NYC', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
print("🎨 Cleaning and Standardizing Vehicle Colors...")

# Create a color mapping dictionary
color_mapping = {
    # Black variations
    'BLK': 'BLACK',
    'BLACK': 'BLACK',
    'BK': 'BLACK',
    'BL': 'BLACK',
    
    # White variations
    'WHI': 'WHITE',
    'WHITE': 'WHITE',
    'WHT': 'WHITE',
    'WH': 'WHITE',
    
    # Gray/Grey variations
    'GY': 'GREY',
    'GREY': 'GREY',
    'GRAY': 'GREY',
    'GRY': 'GREY',
    'SLV': 'SILVER',
    'SILVER': 'SILVER',
    'SILVE': 'SILVER',
    'SLVR': 'SILVER',
    
    # Blue variations
    'BLU': 'BLUE',
    'BLUE': 'BLUE',
    'LTBLU': 'LIGHT BLUE',
    'DKBLU': 'DARK BLUE',
    
    # Red variations
    'RED': 'RED',
    'RD': 'RED',
    
    # Green variations
    'GRN': 'GREEN',
    'GREEN': 'GREEN',
    'GR': 'GREEN',
    
    # Brown variations
    'BRO': 'BROWN',
    'BR': 'BROWN',
    'BROWN': 'BROWN',
    'TAN': 'BROWN',
    'TN': 'BROWN',
    'BEIGE': 'BROWN',
    
    # Other common colors
    'GLD': 'GOLD',
    'GL': 'GOLD',
    'GOLD': 'GOLD',
    'ORG': 'ORANGE',
    'ORANGE': 'ORANGE',
    'PNK': 'PINK',
    'PINK': 'PINK',
    'PUR': 'PURPLE',
    'PURPLE': 'PURPLE',
    'YEL': 'YELLOW',
    'YW': 'YELLOW',
    'YELLOW': 'YELLOW',
    
    # Multi-color or special
    'MULTI': 'MULTI-COLOR',
    'MULTICOLOR': 'MULTI-COLOR',
}

# Apply the mapping to create a standardized color column
df_clean['vehicle_color_standard'] = df_clean['vehicle_color'].map(color_mapping)

# For colors not in the mapping, keep the original but clean it
df_clean['vehicle_color_standard'] = df_clean['vehicle_color_standard'].fillna(df_clean['vehicle_color'])

# Remove any remaining NaN or unknown values
df_clean['vehicle_color_standard'] = df_clean['vehicle_color_standard'].replace(['NAN', 'UNKNOWN', '', ' '], pd.NA)

# Show the results of cleaning
print("\n📊 Before Cleaning - Top 10 Colors:")
before_counts = df_clean['vehicle_color'].value_counts().head(10)
print(before_counts)

print("\n📊 After Cleaning & Grouping - Top 10 Colors:")
after_counts = df_clean['vehicle_color_standard'].value_counts().head(10)
print(after_counts)

print("\n✅ Color standardization complete!")

In [ ]:
print("🎨 Top Standardized Vehicle Colors Receiving Tickets:")

# Filter out null/unknown values
valid_colors = df_clean[df_clean['vehicle_color_standard'].notna()]
valid_colors = valid_colors[valid_colors['vehicle_color_standard'] != 'UNKNOWN']

top_colors_standard = valid_colors['vehicle_color_standard'].value_counts().head(10)
top_colors_standard

# Create a dictionary that maps each color name to its bar color
bar_color_map = {
    'BLACK': 'black',
    'GREY': 'gray',
    'SILVER': 'silver',
    'WHITE': 'whitesmoke',
    'BLUE': 'blue',
    'RED': 'red',
    'GREEN': 'green',
    'BROWN': 'brown',
    'GOLD': 'gold',
    'ORANGE': 'orange',
    'YELLOW': 'yellow',
    'PURPLE': 'purple',
    'PINK': 'pink',
    'MULTI-COLOR': 'cyan'
}

# Get the actual color for each bar based on its name
bar_colors = [bar_color_map.get(color, 'skyblue') for color in top_colors_standard.index]

# Create the visualization
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(top_colors_standard)), top_colors_standard.values, color=bar_colors, edgecolor='black')
plt.xticks(range(len(top_colors_standard)), top_colors_standard.index, rotation=45, ha='right')
plt.xlabel('Vehicle Color', fontsize=12)
plt.ylabel('Number of Tickets', fontsize=12)
plt.title('Top 10 Vehicle Colors Receiving Parking Tickets (Standardized)', fontsize=14, fontweight='bold')

# Add value labels on top of bars
for i, v in enumerate(top_colors_standard.values):
    plt.text(i, v + (v * 0.01), f'{v:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Show percentage breakdown
print("\n📊 Percentage Breakdown of Top Colors:")
total = len(valid_colors)
for color, count in top_colors_standard.items():
    pct = (count / total) * 100
    print(f"   {color}: {count:,} tickets ({pct:.1f}%)")

In [ ]:
# Show examples of how specific colors were cleaned
print("🔍 Examples of Color Standardization:")
example_colors = ['GY', 'GREY', 'GRAY', 'GRY', 'BLK', 'BLACK', 'WHI', 'WHITE', 'SLV', 'SILVER']

for color in example_colors:
    standard = color_mapping.get(color, color)
    print(f"   '{color}' → '{standard}'")

# Show the most common original colors that became GREY
print("\n📋 Original colors that mapped to 'GREY':")
grey_originals = df_clean[df_clean['vehicle_color_standard'] == 'GREY']['vehicle_color'].value_counts().head(10)
print(grey_originals)

In [ ]:
print("🎨 Top Standardized Vehicle Colors Receiving Tickets:")

# Filter out null/unknown values
valid_colors = df_clean[df_clean['vehicle_color_standard'].notna()]
valid_colors = valid_colors[valid_colors['vehicle_color_standard'] != 'UNKNOWN']

top_colors_standard = valid_colors['vehicle_color_standard'].value_counts().head(10)

# Create a color map for the bars
bar_colors = {
    'BLACK': 'black',
    'WHITE': 'lightgray',
    'GREY': 'gray',
    'SILVER': 'silver',
    'BLUE': 'blue',
    'RED': 'red',
    'GREEN': 'green',
    'BROWN': 'brown',
    'GOLD': 'gold',
    'ORANGE': 'orange'
}

# Get colors for the top 10
colors_to_use = [bar_colors.get(color, 'skyblue') for color in top_colors_standard.index]

# Create the visualization
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(top_colors_standard)), top_colors_standard.values, color=colors_to_use, edgecolor='black')
plt.xticks(range(len(top_colors_standard)), top_colors_standard.index, rotation=45, ha='right')
plt.xlabel('Vehicle Color', fontsize=12)
plt.ylabel('Number of Tickets', fontsize=12)
plt.title('Top 10 Vehicle Colors Receiving Parking Tickets', fontsize=14, fontweight='bold')

# Add value labels on top of bars
for i, v in enumerate(top_colors_standard.values):
    plt.text(i, v + (v * 0.01), f'{v:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Show percentage breakdown
print("\n📊 Percentage Breakdown of Top Colors:")
total = len(valid_colors)
for color, count in top_colors_standard.items():
    pct = (count / total) * 100
    print(f"   {color}: {count:,} tickets ({pct:.1f}%)")

In [ ]:
print("🚗 Top Vehicle Makes Receiving Tickets:")

valid_makes = df_clean[df_clean['vehicle_make'].notna()]
valid_makes = valid_makes[valid_makes['vehicle_make'] != 'UNKNOWN']
top_makes = valid_makes['vehicle_make'].value_counts().head(10)
top_makes

In [ ]:
print("🚗 Top Vehicle Body Types Receiving Tickets")
print("(SUBN=Suburban, VAN=Van, 4DSD=4-Door Sedan, etc.)")

body_types = df_clean['vehicle_body_type'].value_counts().head(10)

plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(body_types)), body_types.values, color='teal', edgecolor='black')
plt.xlabel('Vehicle Body Type', fontsize=12)
plt.ylabel('Number of Tickets', fontsize=12)
plt.title('Top 10 Vehicle Body Types for Parking Violations', fontsize=14, fontweight='bold')
plt.xticks(range(len(body_types)), body_types.index, rotation=45, ha='right')

for i, (bar, v) in enumerate(zip(bars, body_types.values)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000, 
             f'{v:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Body Type Breakdown:")
for body, count in body_types.items():
    pct = (count / len(df_clean)) * 100
    print(f"   {body}: {count:,} tickets ({pct:.1f}%)")

In [ ]:
print("🗺️ Parking Violations by NYC Borough (Standardized)")

# Create a comprehensive mapping for boroughs
borough_mapping = {
    # Manhattan
    'NY': 'Manhattan',
    'MN': 'Manhattan',
    'MANHATTAN': 'Manhattan',
    'NEW YORK': 'Manhattan',
    
    # Brooklyn
    'BK': 'Brooklyn',
    'K': 'Brooklyn',
    'KINGS': 'Brooklyn',
    'BROOKLYN': 'Brooklyn',
    'Kings': 'Brooklyn',
    
    # Queens
    'QN': 'Queens',
    'Q': 'Queens',
    'QNS': 'Queens',
    'Queens': 'Queens',
    'Qns': 'Queens',
    
    # Bronx
    'BX': 'Bronx',
    'BRONX': 'Bronx',
    
    # Staten Island
    'SI': 'Staten Island',
    'R': 'Staten Island',
    'RICH': 'Staten Island',
    'ST': 'Staten Island',
}

# Apply the mapping
df_clean['borough'] = df_clean['violation_county'].map(borough_mapping)

# Fill any unmapped values with 'Other' and see what's left
unmapped = df_clean[df_clean['borough'].isna()]['violation_county'].unique()
if len(unmapped) > 0:
    print(f"⚠️ Unmapped county codes found: {unmapped}")
    df_clean['borough'] = df_clean['borough'].fillna('Other')

# Calculate borough totals
borough_counts = df_clean['borough'].value_counts()

# Create the pie chart
plt.figure(figsize=(8, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
plt.pie(borough_counts.values, labels=borough_counts.index, autopct='%1.1f%%', 
        startangle=90, colors=colors, explode=[0.02] * len(borough_counts))
plt.title('Parking Violations by NYC Borough (Standardized)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/visuals/borough_breakdown.png', dpi=150, bbox_inches='tight')

plt.show()

print("\n📊 Standardized Borough Breakdown:")
for borough, count in borough_counts.items():
    pct = (count / len(df_clean)) * 100
    print(f"   {borough}: {count:,} tickets ({pct:.1f}%)")

In [ ]:
# Show the mapping results
print("🔍 Verification of Borough Mapping:")
print("\nOriginal codes and their mapped boroughs:")
sample_codes = ['NY', 'MN', 'BK', 'K', 'KINGS', 'QN', 'Q', 'QNS', 'BX', 'BRONX', 'SI', 'R', 'RICH', 'ST']

for code in sample_codes:
    if code in df_clean['violation_county'].values:
        count = len(df_clean[df_clean['violation_county'] == code])
        mapped = borough_mapping.get(code, 'Other')
        print(f"   '{code}' → {mapped}: {count:,} tickets")

In [ ]:
print("🎨 Top Standardized Vehicle Colors Receiving Tickets:")

# Filter out null/unknown values
valid_colors = df_clean[df_clean['vehicle_color_standard'].notna()]
valid_colors = valid_colors[valid_colors['vehicle_color_standard'] != 'UNKNOWN']

top_colors_standard = valid_colors['vehicle_color_standard'].value_counts().head(10)

# Create a color map for the bars
bar_colors = {
    'BLACK': 'black',
    'WHITE': 'lightgray',
    'GREY': 'gray',
    'SILVER': 'silver',
    'BLUE': 'blue',
    'RED': 'red',
    'GREEN': 'green',
    'BROWN': 'brown',
    'GOLD': 'gold',
    'ORANGE': 'orange'
}

# Get colors for the top 10
colors_to_use = [bar_colors.get(color, 'skyblue') for color in top_colors_standard.index]

# Create the visualization
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(top_colors_standard)), top_colors_standard.values, color=colors_to_use, edgecolor='black')
plt.xticks(range(len(top_colors_standard)), top_colors_standard.index, rotation=45, ha='right')
plt.xlabel('Vehicle Color', fontsize=12)
plt.ylabel('Number of Tickets', fontsize=12)
plt.title('Top 10 Vehicle Colors Receiving Parking Tickets', fontsize=14, fontweight='bold')

# Add value labels on top of bars
for i, v in enumerate(top_colors_standard.values):
    plt.text(i, v + (v * 0.01), f'{v:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../data/visuals/top10_vehicle_colors.png', dpi=150, bbox_inches='tight')

plt.show()

# Show percentage breakdown
print("\n📊 Percentage Breakdown of Top Colors:")
total = len(valid_colors)
for color, count in top_colors_standard.items():
    pct = (count / total) * 100
    print(f"   {color}: {count:,} tickets ({pct:.1f}%)")

In [ ]:
print("📅 Vehicle Year Distribution:")

if 'vehicle_year' in df_clean.columns:
    # Filter reasonable years (1980 to current year)
    current_year = pd.Timestamp.now().year
    valid_years = df_clean[(df_clean['vehicle_year'] >= 1980) & (df_clean['vehicle_year'] <= current_year)]
    
    top_years = valid_years['vehicle_year'].value_counts().head(15).sort_index()
    
    plt.figure(figsize=(12, 6))
    plt.bar(top_years.index.astype(str), top_years.values, color='teal', edgecolor='black')
    plt.xlabel('Vehicle Year', fontsize=12)
    plt.ylabel('Number of Tickets', fontsize=12)
    plt.title('Most Common Vehicle Years Receiving Tickets', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../data/visuals/vehicle_years.png', dpi=150, bbox_inches='tight')
    
    plt.show()
    
    print(f"Most common vehicle year: {top_years.idxmax()} ({top_years.max():,} tickets)")

In [ ]:
print("🚗 Top Violation by Vehicle Make:")

# Get top 5 vehicle makes
top_makes = df_clean['vehicle_make'].value_counts().head(5).index

# For each top make, find their most common violation
for make in top_makes:
    make_data = df_clean[df_clean['vehicle_make'] == make]
    top_violation = make_data['violation_description'].value_counts().head(1)
    if len(top_violation) > 0:
        print(f"   {make}: {top_violation.index[0][:50]} ({top_violation.values[0]:,} tickets)")

In [ ]:
if 'violation_time' in df_clean.columns:
    # Extract hour from violation time
    df_clean['hour'] = df_clean['violation_time'].astype(str).str[:2]
    df_clean['hour'] = pd.to_numeric(df_clean['hour'], errors='coerce')
    
    # Filter valid hours (0-23)
    hour_data = df_clean[df_clean['hour'].between(0, 23)]
    
    if len(hour_data) > 0:
        hour_counts = hour_data['hour'].value_counts().sort_index()
        
        plt.figure(figsize=(12, 6))
        plt.bar(hour_counts.index, hour_counts.values, color='green', edgecolor='black', alpha=0.7)
        plt.xlabel('Hour of Day', fontsize=12)
        plt.ylabel('Number of Tickets', fontsize=12)
        plt.title('Parking Tickets by Time of Day', fontsize=14, fontweight='bold')
        plt.xticks(range(0, 24))
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print("\n📊 Busiest Hours for Tickets:")
        print(hour_counts.head(5))
    else:
        print("⚠️ No valid hour data found")
else:
    print("⚠️ 'violation_time' column not found in dataset")

In [ ]:
print("⏰ Time of Day Heatmap:")

# Extract hour and day of week
df_clean['hour'] = df_clean['violation_time'].astype(str).str[:2]
df_clean['hour'] = pd.to_numeric(df_clean['hour'], errors='coerce')
df_clean['day_of_week'] = df_clean['issue_date'].dt.day_name()

# Filter valid hours
hour_day_data = df_clean.dropna(subset=['hour', 'day_of_week'])
hour_day_data = hour_day_data[hour_day_data['hour'].between(0, 23)]

# Create pivot table
heatmap_data = hour_day_data.groupby(['hour', 'day_of_week']).size().unstack()

# Order days correctly
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data[day_order]

plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, cmap='viridis', annot=False, cbar_kws={'label': 'Number of Tickets'})
plt.title('Parking Violations: Hour vs Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Hour of Day', fontsize=12)
plt.tight_layout()
plt.savefig('../data/visuals/time_of_day_heatmap.png', dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
# Create year and month columns
df_clean['year'] = df_clean['issue_date'].dt.year
df_clean['month'] = df_clean['issue_date'].dt.month

# Create pivot table
ticket_heatmap = df_clean.groupby(['year', 'month']).size().unstack()

plt.figure(figsize=(14, 8))
sns.heatmap(ticket_heatmap, cmap='YlOrRd', annot=False, cbar_kws={'label': 'Number of Tickets'})
plt.title('Parking Tickets Heatmap: Year vs Month', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Year', fontsize=12)

# Rename months
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
plt.xticks(range(12), month_names, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
print("⏰ Analysis of Parking Restriction Hours")

# Clean and extract hours from from_hours_in_effect
if 'from_hours_in_effect' in df_clean.columns and df_clean['from_hours_in_effect'].notna().sum() > 10000:
    # Extract hour from time strings like "0730A", "0400P", "ALL"
    def extract_hour(time_str):
        if pd.isna(time_str) or time_str == 'ALL':
            return None
        time_str = str(time_str)
        # Extract digits
        import re
        match = re.search(r'(\d{1,2})', time_str)
        if match:
            hour = int(match.group(1))
            # Convert to 24-hour format if PM
            if 'P' in time_str and hour < 12:
                hour += 12
            return hour
        return None
    
    df_clean['restriction_start_hour'] = df_clean['from_hours_in_effect'].apply(extract_hour)
    valid_hours = df_clean['restriction_start_hour'].dropna()
    valid_hours = valid_hours[valid_hours.between(0, 23)]
    
    if len(valid_hours) > 0:
        hour_counts = valid_hours.value_counts().sort_index()
        
        plt.figure(figsize=(12, 6))
        plt.bar(hour_counts.index, hour_counts.values, color='purple', edgecolor='black')
        plt.xlabel('Hour Parking Restriction Begins', fontsize=12)
        plt.ylabel('Number of Violations', fontsize=12)
        plt.title('Parking Violations by Restriction Start Time', fontsize=14, fontweight='bold')
        plt.xticks(range(0, 24))
        plt.grid(axis='y', alpha=0.3)
        
        # Highlight peak hour
        peak_hour = hour_counts.idxmax()
        plt.axvline(x=peak_hour, color='red', linestyle='--', alpha=0.7, label=f'Peak: {peak_hour}:00')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig('../data/visuals/restriction_start_time.png', dpi=150, bbox_inches='tight')
        
        plt.show()
        
        print(f"\n📊 Peak restriction hour: {peak_hour}:00 ({hour_counts.max():,} violations)")
    else:
        print("ℹ️ Could not parse hour data from from_hours_in_effect")
else:
    print("ℹ️ from_hours_in_effect column has limited data")

In [ ]:
# Check all column names in your dataframe
print("All available columns:")
for i, col in enumerate(df_clean.columns):
    print(f"  {i+1}. {col}")

# Look for any columns related to fines or amounts
print("\n🔍 Columns that might contain fine information:")
fine_related = [col for col in df_clean.columns if 'fine' in col.lower() or 'amount' in col.lower() or 'penalty' in col.lower()]
if fine_related:
    for col in fine_related:
        print(f"  - {col}")
else:
    print("  No fine-related columns found with 'fine', 'amount', or 'penalty' in the name")

# Also check the raw dataframe before cleaning
print("\n🔍 Raw dataframe columns (first 20):")
for i, col in enumerate(df.columns[:20]):
    print(f"  {i+1}. {col}")

In [ ]:
print("🌸 Seasonal Analysis:")

# Extract month name
df_clean['month_name'] = df_clean['issue_date'].dt.month_name()

# Order months correctly
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

monthly_seasonal = df_clean['month_name'].value_counts()
monthly_seasonal = monthly_seasonal.reindex(month_order)

plt.figure(figsize=(12, 6))
plt.bar(monthly_seasonal.index, monthly_seasonal.values, color='skyblue', edgecolor='black')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Number of Tickets', fontsize=12)
plt.title('Seasonal Pattern of Parking Violations', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/visuals/seasonal_pattern.png', dpi=150, bbox_inches='tight')

plt.show()

# Find busiest and quietest months
print(f"Busiest month: {monthly_seasonal.idxmax()} ({monthly_seasonal.max():,} tickets)")
print(f"Quietest month: {monthly_seasonal.idxmin()} ({monthly_seasonal.min():,} tickets)")

In [ ]:
print("🗺️ Vehicle Registration State Analysis:")

if 'registration_state' in df_clean.columns:
    state_counts = df_clean['registration_state'].value_counts().head(15)
    
    plt.figure(figsize=(12, 6))
    state_counts.plot(kind='bar', color='navy', edgecolor='black')
    plt.xlabel('State', fontsize=12)
    plt.ylabel('Number of Tickets', fontsize=12)
    plt.title('Parking Violations by Vehicle Registration State', fontsize=14, fontweight='bold')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('../data/visuals/vehicle_registration_states.png', dpi=150, bbox_inches='tight')
    
    plt.show()
    
    ny_count = state_counts.get('NY', 0)
    non_ny_count = len(df_clean) - ny_count
    print(f"NY registered vehicles: {ny_count:,} tickets ({ny_count/len(df_clean)*100:.1f}%)")
    print(f"Out-of-state vehicles: {non_ny_count:,} tickets ({non_ny_count/len(df_clean)*100:.1f}%)")

In [ ]:
print("=" * 60)
print("📊 FINAL SUMMARY - NYC PARKING VIOLATIONS")
print("=" * 60)

# Recalculate top violations if needed
if 'top_violations' not in locals():
    top_violations = df_clean.groupby(['violation_code', 'violation_description']).size().reset_index(name='ticket_count')
    top_violations = top_violations.sort_values('ticket_count', ascending=False).head(1)

summary = {
    "Total Tickets": f"{len(df_clean):,}",
    "Date Range": f"{df_clean['issue_date'].min().date()} to {df_clean['issue_date'].max().date()}",
    "Unique Vehicles (Plates)": f"{df_clean['plate_id'].nunique():,}",
    "Unique Makes": df_clean['vehicle_make'].nunique(),
    "Unique Colors": df_clean['vehicle_color_standard'].nunique() if 'vehicle_color_standard' in df_clean.columns else df_clean['vehicle_color'].nunique(),
    "Unique Violations": df_clean['violation_code'].nunique(),
    "Unique Precincts": df_clean['violation_precinct'].nunique(),
    "Top Violation": top_violations.iloc[0]['violation_description'] if len(top_violations) > 0 else "N/A",
    "Top Vehicle Make": df_clean['vehicle_make'].value_counts().index[0] if 'vehicle_make' in df_clean.columns else "N/A"
}

for key, value in summary.items():
    print(f"{key:25}: {value}")

print("\n" + "=" * 60)
print("✅ Analysis complete! Your data is ready for merging with a second dataset.")
print("=" * 60)

In [ ]:
# Save cleaned data for future use
df_clean.to_csv('../data/parking_violations_cleaned.csv', index=False)
print("✅ Cleaned data saved to 'parking_violations_cleaned.csv'")

# Save a smaller sample for quick testing (10,000 rows)
df_clean.head(10000).to_csv('../data/parking_violations_sample.csv', index=False)
print("✅ Sample data (10,000 rows) saved to '../data/parking_violations_sample.csv'")

print(f"\n📁 Final cleaned dataset shape: {df_clean.shape}")
print(f"📅 Date range: {df_clean['issue_date'].min()} to {df_clean['issue_date'].max()}")

In [ ]:
print("=" * 60)
print("LOADING AND CLEANING FINES DATA")
print("=" * 60)

# Load your fines CSV
df_fines = pd.read_csv("../data/fines_extracted_fixed.csv")  # Use your actual filename

# Rename columns if needed
df_fines.columns = ['violation_code', 'violation_description', 'fine_amount']

print(f"📊 Raw fines data: {len(df_fines)} rows")
print(f"📋 Columns: {df_fines.columns.tolist()}")

# Check unique values in fine_amount column
print(f"\n🔍 Unique values in Fine Amount column:")
print(df_fines['fine_amount'].value_counts().head(20))

# Clean the fine amount column
def clean_fine_amount(value):
    """Convert fine amount to float, handling non-numeric values"""
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).strip()
    
    # Handle special cases
    if value_str.lower() == 'vary':
        return np.nan  # or use a default like 100? We'll use NaN for now
    if value_str == '$0' or value_str == '0':
        return 0.0
    
    # Remove $ sign and any spaces
    cleaned = value_str.replace('$', '').replace(',', '').strip()
    
    # Try to convert to float
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

# Apply the cleaning function
df_fines['fine_amount_clean'] = df_fines['fine_amount'].apply(clean_fine_amount)

# Remove rows where fine amount is NaN (including 'vary')
df_fines_clean = df_fines.dropna(subset=['fine_amount_clean'])

print(f"\n✅ After cleaning:")
print(f"   Total rows: {len(df_fines)}")
print(f"   Rows with valid fine amounts: {len(df_fines_clean)}")
print(f"   Removed {len(df_fines) - len(df_fines_clean)} rows with 'vary' or invalid values")

print(f"\n📊 Fine amount statistics:")
print(df_fines_clean['fine_amount_clean'].describe())

# Preview the cleaned data
print(f"\n📋 Preview of cleaned fines data:")
print(df_fines_clean[['violation_code', 'violation_description', 'fine_amount_clean']].head(10))

In [ ]:
# Standardize violation codes
df_clean['violation_code'] = df_clean['violation_code'].astype(str).str.strip()
df_fines['violation_code'] = df_fines['violation_code'].astype(str).str.strip()

# Merge
df_merged = df_clean.merge(
    df_fines[['violation_code', 'fine_amount_clean', 'violation_description']],
    on='violation_code',
    how='left'
)

print(f"\n✅ Merge complete!")
print(f"   Total rows: {len(df_merged):,}")
print(f"   Rows with fine amounts: {df_merged['fine_amount_clean'].notna().sum():,} ({df_merged['fine_amount_clean'].notna().sum()/len(df_merged)*100:.1f}%)")

In [ ]:
print("=" * 60)
print("MERGE VERIFICATION")
print("=" * 60)

print(f"📊 Merged data shape: {df_merged.shape}")
print(f"   Rows: {len(df_merged):,}")
print(f"   Columns: {len(df_merged.columns)}")

print(f"\n📋 Key columns in merged data:")
key_cols = ['violation_code', 'violation_description', 'fine_amount_clean', 'borough', 'issue_date', 'vehicle_make', 'vehicle_color']
for col in key_cols:
    if col in df_merged.columns:
        print(f"   ✅ {col}")
    else:
        print(f"   ❌ {col} - MISSING")

print(f"\n💰 Fine amount coverage:")
fine_coverage = df_merged['fine_amount_clean'].notna().sum()
print(f"   {fine_coverage:,} / {len(df_merged):,} rows have fine amounts ({fine_coverage/len(df_merged)*100:.1f}%)")

print(f"\n💵 Fine amount statistics:")
print(df_merged['fine_amount_clean'].describe())

In [ ]:
print("=" * 60)
print("FINDING VIOLATION DESCRIPTION COLUMN")
print("=" * 60)

# Show all columns
print("\n📋 All columns in merged data:")
for i, col in enumerate(df_merged.columns):
    print(f"   {i+1}. {col}")

# Look for any column with 'description' or 'violation' in the name
desc_cols = [col for col in df_merged.columns if 'description' in col.lower() or 'violation' in col.lower()]
print(f"\n🔍 Potential description columns: {desc_cols}")

# If you have the original violation_description from parking data
if 'violation_description' in df_merged.columns:
    print("\n✅ Found 'violation_description' column!")
    print(df_merged['violation_description'].head(3))
elif 'Violation Description' in df_merged.columns:
    print("\n✅ Found 'Violation Description' column!")
    df_merged.rename(columns={'Violation Description': 'violation_description'}, inplace=True)
else:
    print("\n⚠️ No description column found. You may need to re-merge or check your fines file.")

In [ ]:
print("=" * 60)
print("CLEANING VIOLATION DESCRIPTION COLUMNS")
print("=" * 60)

# See what each description column contains
print("\n📋 violation_description_x (from parking data):")
print(df_merged['violation_description_x'].head(3))

print("\n📋 violation_description_y (from fines data):")
print(df_merged['violation_description_y'].head(3))

# Create a single description column
# Prefer the fines description (more detailed) but fall back to parking data
df_merged['violation_description'] = df_merged['violation_description_y'].fillna(df_merged['violation_description_x'])

# Drop the old description columns
df_merged = df_merged.drop(columns=['violation_description_x', 'violation_description_y'])

print(f"\n✅ Created single 'violation_description' column")
print(f"   Coverage: {df_merged['violation_description'].notna().sum():,} / {len(df_merged):,} rows")
print(f"\n📋 Sample descriptions:")
print(df_merged['violation_description'].head(10))

In [ ]:
print("\n" + "=" * 60)
print("BOX PLOT: Fine Amounts by Borough")
print("=" * 60)

# Prepare data
box_data = df_merged[df_merged['fine_amount_clean'].notna()].copy()
box_data = box_data[box_data['borough'] != 'Other']

# Sample for performance
if len(box_data) > 100000:
    box_data = box_data.sample(n=100000, random_state=42)

plt.figure(figsize=(12, 7))

boroughs = box_data['borough'].unique()
fine_values = [box_data[box_data['borough'] == b]['fine_amount_clean'].values for b in boroughs]

bp = plt.boxplot(fine_values, labels=boroughs, patch_artist=True,
                  medianprops=dict(linewidth=2, color='darkred'),
                  whiskerprops=dict(linewidth=1.5),
                  flierprops=dict(marker='o', markersize=3, alpha=0.3))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
for patch, color in zip(bp['boxes'], colors[:len(boroughs)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.xlabel('Borough', fontsize=14, fontweight='bold')
plt.ylabel('Fine Amount ($)', fontsize=14, fontweight='bold')
plt.title('Distribution of Parking Fine Amounts by Borough', fontsize=16, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Add median values
for i, borough in enumerate(boroughs):
    median = np.median(fine_values[i])
    plt.text(i + 1, median + 5, f'${median:.0f}', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.savefig('../data/visuals/Median_Fine_by_Borough.png', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

print("\n📊 Median Fine by Borough:")
for i, borough in enumerate(boroughs):
    median = np.median(fine_values[i])
    q1 = np.percentile(fine_values[i], 25)
    q3 = np.percentile(fine_values[i], 75)
    print(f"   {borough}: ${median:.0f} (IQR: ${q1:.0f}-${q3:.0f})")

In [ ]:
print("\n" + "=" * 60)
print("CORRELATION MATRIX")
print("=" * 60)

# Select numeric columns
numeric_cols = ['violation_code', 'vehicle_year', 'violation_precinct', 'fine_amount_clean']
corr_data = df_merged[numeric_cols].dropna()

if len(corr_data) > 50000:
    corr_data = corr_data.sample(n=50000, random_state=42)

corr_matrix = corr_data.corr()

plt.figure(figsize=(10, 8))
im = plt.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

cbar = plt.colorbar(im)
cbar.set_label('Correlation Coefficient', fontsize=12, fontweight='bold')

plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right', fontsize=11)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns, fontsize=11)

for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        plt.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                ha="center", va="center", 
                color="black" if abs(corr_matrix.iloc[i, j]) < 0.5 else "white",
                fontsize=10, fontweight='bold')

plt.title('Correlation Matrix: Parking Violation Variables', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n📊 Correlations with Fine Amount:")
fine_corrs = corr_matrix['fine_amount_clean'].drop('fine_amount_clean')
for var, corr in fine_corrs.items():
    strength = "strong" if abs(corr) > 0.5 else "moderate" if abs(corr) > 0.3 else "weak"
    direction = "positive" if corr > 0 else "negative"
    print(f"   {var}: {corr:.3f} ({strength} {direction})")

In [ ]:
print("\n" + "=" * 60)
print("TOP 10 MOST EXPENSIVE VIOLATIONS")
print("=" * 60)

# Get unique violations with their fines
violation_fines = df_merged[df_merged['fine_amount_clean'].notna()][['violation_code', 'violation_description', 'fine_amount_clean']].drop_duplicates()
top_fines = violation_fines.sort_values('fine_amount_clean', ascending=False).head(10)

plt.figure(figsize=(12, 6))
bars = plt.barh(range(len(top_fines)), top_fines['fine_amount_clean'], color='darkred', edgecolor='black')
plt.yticks(range(len(top_fines)), top_fines['violation_description'].str[:50])
plt.xlabel('Fine Amount ($)', fontsize=12, fontweight='bold')
plt.title('Top 10 Most Expensive Parking Violations', fontsize=14, fontweight='bold')

for i, (bar, val) in enumerate(zip(bars, top_fines['fine_amount_clean'])):
    plt.text(val + 5, bar.get_y() + bar.get_height()/2, f'${int(val)}', va='center', fontsize=10)


plt.tight_layout()
plt.show()

print(top_fines[['violation_code', 'violation_description', 'fine_amount_clean']])

In [ ]:
print("=" * 60)
print("DIAGNOSING REVENUE DATA ISSUE")
print("=" * 60)

# Check if fine_amount_clean has data
print(f"\n1. Fine amount coverage:")
fine_count = df_merged['fine_amount_clean'].notna().sum()
print(f"   Rows with fine amounts: {fine_count:,} / {len(df_merged):,} ({fine_count/len(df_merged)*100:.1f}%)")

# Check borough distribution
print(f"\n2. Borough distribution:")
borough_counts = df_merged['borough'].value_counts()
print(borough_counts)

# Check combined data (fine amounts by borough)
print(f"\n3. Fine amounts by borough (non-null):")
fine_by_borough = df_merged[df_merged['fine_amount_clean'].notna()].groupby('borough')['fine_amount_clean'].sum()
print(fine_by_borough)

# Check if 'Other' is filtering correctly
print(f"\n4. Checking for 'Other' borough:")
other_count = (df_merged['borough'] == 'Other').sum()
print(f"   'Other' rows: {other_count:,}")


In [ ]:
print("\n" + "=" * 60)
print("TOTAL REVENUE BY BOROUGH")
print("=" * 60)

# Calculate revenue by borough (excluding 'Other')
revenue_by_borough = df_merged[df_merged['fine_amount_clean'].notna()].groupby('borough')['fine_amount_clean'].sum().sort_values(ascending=False)
revenue_by_borough = revenue_by_borough[revenue_by_borough.index != 'Other']

print(f"Revenue data:\n{revenue_by_borough}")

# Create the plot
plt.figure(figsize=(10, 6))
bars = plt.bar(revenue_by_borough.index, revenue_by_borough.values / 1_000_000, 
               color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'],
               edgecolor='black')

plt.xlabel('Borough', fontsize=12, fontweight='bold')
plt.ylabel('Total Revenue (Millions of $)', fontsize=12, fontweight='bold')
plt.title('Estimated Total Parking Fine Revenue by Borough', fontsize=14, fontweight='bold')

# Add value labels on top of bars
for bar, val in zip(bars, revenue_by_borough.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (val/1_000_000 * 0.02), 
             f'${val/1_000_000:.1f}M', ha='center', va='bottom', fontsize=10)
    
plt.tight_layout()
plt.show()

print("\n💰 Revenue by Borough:")
for borough, revenue in revenue_by_borough.items():
    print(f"   {borough}: ${revenue/1_000_000:.2f} Million")

In [ ]:
# Save the final cleaned dataset
df_merged.to_csv("../data/parking_violations_final.csv", index=False)
print("\n✅ Final dataset saved to 'parking_violations_final.csv'")
print(f"   Shape: {df_merged.shape}")
print(f"   Columns: {df_merged.columns.tolist()}")

In [ ]:
print("=" * 60)
print("README NUMBERS EXTRACTOR")
print("=" * 60)

# Basic counts
print(f"Total Tickets: {len(df_merged):,}")
print(f"Date Range: {df_merged['issue_date'].min().date()} to {df_merged['issue_date'].max().date()}")

# Revenue calculations
revenue_by_borough = df_merged[df_merged['fine_amount_clean'].notna()].groupby('borough')['fine_amount_clean'].sum().sort_values(ascending=False)
revenue_by_borough = revenue_by_borough[revenue_by_borough.index != 'Other']
total_revenue = revenue_by_borough.sum()

print(f"\nTotal Revenue: ${total_revenue/1_000_000:.1f} Million")

# Borough revenue
for borough, revenue in revenue_by_borough.items():
    pct = (revenue / total_revenue) * 100
    print(f"{borough}: ${revenue/1_000_000:.1f}M ({pct:.1f}%)")

# Top violations (most expensive)
print("\nTop 5 Most Expensive Violations:")
expensive = df_merged[df_merged['fine_amount_clean'].notna()][['violation_code', 'violation_description', 'fine_amount_clean']].drop_duplicates()
expensive = expensive.sort_values('fine_amount_clean', ascending=False).head(5)
for _, row in expensive.iterrows():
    print(f"  {row['violation_description'][:50]}: ${row['fine_amount_clean']:.0f}")

# Top violations (most common)
print("\nTop 5 Most Common Violations:")
common = df_merged['violation_description'].value_counts().head(5)
for desc, count in common.items():
    print(f"  {desc[:50]}: {count:,} tickets")

# Vehicle analysis
print(f"\nTop Vehicle Make: {df_merged['vehicle_make'].value_counts().index[0]}")
print(f"Top Vehicle Color: {df_merged['vehicle_color_standard'].value_counts().index[0] if 'vehicle_color_standard' in df_merged.columns else df_merged['vehicle_color'].value_counts().index[0]}")

# Time patterns
df_merged['hour'] = pd.to_numeric(df_merged['violation_time'].astype(str).str[:2], errors='coerce')
peak_hour = df_merged['hour'].mode()[0]
print(f"Peak Violation Hour: {peak_hour}:00")

# Borough distribution
print("\nBorough Distribution:")
borough_counts = df_merged[df_merged['borough'] != 'Other']['borough'].value_counts()
for borough, count in borough_counts.items():
    pct = (count / len(df_merged)) * 100
    print(f"  {borough}: {count:,} ({pct:.1f}%)")

In [ ]:
# Create a visuals folder
os.makedirs('../data/visuals', exist_ok=True)

# 1. Revenue by Borough
plt.figure(figsize=(10, 6))
revenue_by_borough = df_merged[df_merged['fine_amount_clean'].notna()].groupby('borough')['fine_amount_clean'].sum()
revenue_by_borough = revenue_by_borough[revenue_by_borough.index != 'Other']
bars = plt.bar(revenue_by_borough.index, revenue_by_borough.values / 1_000_000, 
               color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
plt.xlabel('Borough')
plt.ylabel('Revenue (Millions $)')
plt.title('Total Parking Fine Revenue by Borough')
for bar, val in zip(bars, revenue_by_borough.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'${val/1_000_000:.1f}M', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../data/visuals/revenue_by_borough.png', dpi=150, bbox_inches='tight')
plt.close()

# 2. Box Plot
plt.figure(figsize=(12, 7))
box_data = df_merged[df_merged['fine_amount_clean'].notna()].copy()
box_data = box_data[box_data['borough'] != 'Other']
if len(box_data) > 100000:
    box_data = box_data.sample(n=100000)
boroughs = box_data['borough'].unique()
fine_values = [box_data[box_data['borough'] == b]['fine_amount_clean'].values for b in boroughs]
bp = plt.boxplot(fine_values, labels=boroughs, patch_artist=True)
plt.xlabel('Borough')
plt.ylabel('Fine Amount ($)')
plt.title('Distribution of Fine Amounts by Borough')
plt.savefig('../data/visuals/fine_distribution_boxplot.png', dpi=150, bbox_inches='tight')
plt.close()

# 3. Correlation Matrix
numeric_cols = ['violation_code', 'vehicle_year', 'violation_precinct', 'fine_amount_clean']
corr_data = df_merged[numeric_cols].dropna()
if len(corr_data) > 50000:
    corr_data = corr_data.sample(n=50000)
corr_matrix = corr_data.corr()
plt.figure(figsize=(10, 8))
im = plt.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im)
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        plt.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('../data/visuals/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.close()

# 4. Top Violations
top_fines = df_merged[df_merged['fine_amount_clean'].notna()][['violation_description', 'fine_amount_clean']].drop_duplicates()
top_fines = top_fines.sort_values('fine_amount_clean', ascending=False).head(10)
plt.figure(figsize=(12, 6))
plt.barh(range(len(top_fines)), top_fines['fine_amount_clean'], color='darkred')
plt.yticks(range(len(top_fines)), top_fines['violation_description'].str[:50])
plt.xlabel('Fine Amount ($)')
plt.title('Top 10 Most Expensive Violations')
plt.tight_layout()
plt.savefig('../data/visuals/top_violations.png', dpi=150, bbox_inches='tight')
plt.close()

print("✅ All visualizations saved to '../data/visuals/' folder")

Below here is for New API and Datasets being added to improve capstone

In [ ]:
# Aggregate tickets by precinct
tickets_by_precinct = df_merged.groupby('violation_precinct').agg({
    'summons_number': 'count',
    'fine_amount_clean': 'sum',
    'fine_amount_clean': 'mean'
}).reset_index()

tickets_by_precinct.columns = ['precinct', 'ticket_count', 'total_fines', 'avg_fine']
print(tickets_by_precinct.head())